In [101]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [102]:
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from src.model import FiLMResNet2In, flatten_last
from src.normalizer import RunningMeanStd
from src.envpacker import packenv, packenv_batch
import torch
from torch import nn
import torch.nn.functional as F

## 1. Configs

In [103]:
# Training configs
AGENTS = 50     # number of agents
LEARNING_RATE = 1e-3
TRAINING_STEPS = 30000
BATCH_SIZE = 10
DISPLAY_STEP = 1000 # For visualization
TRAIN_STEP_INTERVAL = 2 # Interval of steps between training episodes


# Bewley model parameters
theta = 1 # CRRA
beta = 0.975 # Discount factor
A = 1 # Technology parameter
alpha = 0.33 # Capital share of income
gamma = 2 # Inverse Frisch elasticity
########################################### (MiLF inputs)
r = 0.04 # Interest rate (Return to savings)
w = 1 # Wage rate (Return to labor)
delta = 0.06 # Depreciation rate of capital
TAX_PARAMS = {
    "tax_consumption": 0.065,          # Consumption tax (fixed)
    "tax_income": 0.2,                # Tax on labor income
    "income_tax_elasticity": 0.5,     # Elasticity of labor supply w.r.t. after-tax income
    "saving_tax_elasticity": 0.5,     # Elasticity of savings w.r.t. after-tax income
    "tax_saving": 0.1                 # Tax on interest income
}
###########################################
p = 2.2e-6
q = 0.99

# shock parameters
# log e' = rho_v * log e + sigma_v * epsilon, epsilon ~ N(0,1)
rho_v = 0.95 # persistence of ability shock
sigma_v = 0.2 # std of ability shock


In [104]:
# Bounds of shock 
v_min = math.exp(-2 * sigma_v  / math.sqrt(1-rho_v**2))
v_max = math.exp( 2 * sigma_v  / math.sqrt(1-rho_v**2))

## 2. Helper functions and classes

In [105]:
def mean_across_agents(x): # Since agent number is fix, thus we can use mean instead of sum
    return torch.mean(x, dim=1, keepdim=True)


def calculate_price(a, v, h):
    a_aggregate, l_aggregate_effective = mean_across_agents(a), mean_across_agents(h*v)
    wage = A * (1-alpha) * ((a_aggregate/l_aggregate_effective) ** alpha)
    ret = A * alpha * (a_aggregate/l_aggregate_effective ** alpha)
    return wage, ret

def taxfunc(ibt, abt, taxparams=TAX_PARAMS):
    it = ibt - (1 - taxparams["tax_income"]) * (ibt**(1-taxparams["income_tax_elasticity"])/(1-taxparams["income_tax_elasticity"])) # individual after tax income
    at = abt - (1-taxparams["tax_saving"]/1-taxparams["saving_tax_elasticity"]) * (abt**(1-taxparams["saving_tax_elasticity"])) # individual after tax saving
    return it, at

def calculate_moneydisposable(wage, ret, v, h, a, delta, is_init=False):
    if is_init:
        ibt = wage * h * v   # individual before tax income
    else:
        ibt = wage * h * v + (1-delta+ret) * a   # individual before tax income

    it, at = taxfunc(ibt = ibt, abt=a)
    money_disposable = it + at

    return money_disposable, ibt


def output_transform(a, money_disposable):

    # The a here is saving rate coming from the NN output
    consumption = money_disposable * (1 - a)
    savings = money_disposable * a
    return consumption, savings


def laborfocloss(a, h, ibt, money_disposable, wage, v, taxparams=TAX_PARAMS):

    loss_foc =  -h ** (-gamma) + ((1-a)*money_disposable/(1+taxparams["tax_consumption"])) * \
        (wage * v) * (1 - taxparams["tax_income"]) * (ibt ** (-taxparams["income_tax_elasticity"]))
    
    return torch.abs(loss_foc)

def transition_ability(
    v_prev: torch.Tensor,
    is_superstar_prev: torch.Tensor,
    v_history: torch.Tensor,
    rho_v: float,
    sigma_v: float,
    p: float,
    q: float,
    v_bar: float,
    v_min: float,
    v_max: float
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Calculates the ability value v_t for the next timestep based on the 
    Bewley-Aiyagari model with a normal and a super-star state.

    Args:
        v_prev (torch.Tensor): Ability values from the previous timestep (v_{t-1}).
        is_superstar_prev (torch.Tensor): Boolean tensor indicating which agents 
                                          were in the super-star state.
        v_history (torch.Tensor): A tensor containing the full history of 
                                  ability values for all agents.
        rho_v (float): Persistence parameter for the AR(1) process.
        sigma_v (float): Volatility parameter for the AR(1) process.
        p (float): Probability of transitioning from normal to super-star state.
        q (float): Probability of remaining in the super-star state.
        v_bar (float): Multiplier for super-star ability relative to the average.
        v_min (float): Minimum bound for the ability value.
        v_max (float): Maximum bound for the ability value.

    Returns:
        tuple[torch.Tensor, torch.Tensor]: A tuple containing the new ability 
                                           values (v_t) and the new superstar status.
    """
    num_households = v_prev.shape[0]
    
    # --- 1. Determine State Transitions (Logic Unchanged) ---
    
    transitions = torch.rand(num_households, device=v_prev.device)
    is_superstar_next = is_superstar_prev.clone()
    
    normal_to_superstar_mask = (~is_superstar_prev) & (transitions < p)
    is_superstar_next[normal_to_superstar_mask] = True
    
    superstar_to_normal_mask = is_superstar_prev & (transitions >= q)
    is_superstar_next[superstar_to_normal_mask] = False
    
    # --- 2. Calculate Next Ability v_t ---
    
    v_next = torch.zeros_like(v_prev)
    normal_mask_next = ~is_superstar_next
    superstar_mask_next = is_superstar_next
    
    # --- For agents in the NORMAL state next period (Logic Unchanged) ---
    if normal_mask_next.any():
        shocks = torch.randn(normal_mask_next.sum(), device=v_prev.device)
        log_v_next_normal = rho_v * torch.log(v_prev[normal_mask_next]) + sigma_v * shocks
        v_next_normal = torch.exp(log_v_next_normal)
        v_next[normal_mask_next] = torch.clamp(v_next_normal, min=v_min, max=v_max)

    # --- For agents in the SUPER-STAR state next period (Logic Unchanged) ---
    if superstar_mask_next.any():
        # Calculate the historical average from the provided history tensor.
        # Fallback to the previous period's average if history is empty (e.g., at the first step).
        if v_history is not None and v_history.numel() > 0:
            avg_ability = v_history.mean()
        else:
            avg_ability = v_prev.mean()

        v_next[superstar_mask_next] = v_bar * avg_ability
        
    return v_next, is_superstar_next



In [106]:
# hidden_dim = 64
# num_res_blocks = 2
# output_dim = 3 # a/m, h, \mu


# model = FiLMResNet(
#     AGENTS=AGENTS,
#     hidden_dim=hidden_dim,
#     num_res_blocks=num_res_blocks,
#     dropout=0.1,
#     output_dim=2
# )

In [107]:
# def initial_state(required_batch_size, tax_params=TAX_PARAMS): 
#     # 隨機產生初始資產與儲蓄
#     moneydisposable, savings = np.random.uniform(
#         0.1, 2.0, required_batch_size * AGENTS
#     ).reshape(required_batch_size, AGENTS)

#     # 隨機產生能力值 v，並在批次內標準化
#     v = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(
#         required_batch_size, AGENTS
#     )
#     v = v / np.mean(v, axis=1, keepdims=True)
#     is_superstar = np.zeros((required_batch_size, AGENTS), dtype=bool)

#     # 稅制參數轉為 tensor
#     tax_params = torch.tensor(TAX_PARAMS.values(), dtype=torch.float32)

#     # 以字典形式回傳
#     return {
#         "moneydisposable": torch.tensor(moneydisposable, dtype=torch.float32),
#         "savings": torch.tensor(savings, dtype=torch.float32),
#         "v": torch.tensor(v, dtype=torch.float32),
#         "is_superstar": torch.tensor(is_superstar, dtype=torch.bool),
#         "tax_params": tax_params,
#     }

def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄
    moneydisposable, savings = np.random.uniform(
        0.1, 2.0, required_batch_size * AGENTS
    ).reshape(required_batch_size, AGENTS)

    # 隨機產生能力值 v，並在批次內標準化
    v = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(
        required_batch_size, AGENTS
    )
    v = v / np.mean(v, axis=1, keepdims=True)
    is_superstar = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)

    # 【修正 1】將 tax_params 的形狀從 (5,) 擴展到 (BATCH_SIZE, 5)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 以字典形式回傳
    return {
        # 【修正 2】使用 unsqueeze(-1) 將形狀從 (B, A) 調整為 (B, A, 1)
        "moneydisposable": torch.tensor(moneydisposable, dtype=torch.float32).unsqueeze(-1),
        "savings": torch.tensor(savings, dtype=torch.float32), # savings 不是 packenv 的輸入，可維持原狀
        "v": torch.tensor(v, dtype=torch.float32).unsqueeze(-1),
        "is_superstar": torch.tensor(is_superstar, dtype=torch.bool),
        "tax_params": tax_params,
    }


In [108]:
def initial_state(required_batch_size, tax_params_dict=TAX_PARAMS):
    # 隨機產生初始資產與儲蓄（兩個獨立隨機矩陣）
    moneydisposable = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    savings = np.random.uniform(0.1, 2.0, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)

    # 隨機產生能力值 v，並在批次內標準化
    v = np.random.uniform(v_min, v_max, required_batch_size * AGENTS).reshape(required_batch_size, AGENTS)
    v = v / np.mean(v, axis=1, keepdims=True)
    is_superstar = np.zeros((required_batch_size, AGENTS), dtype=bool)

    # 稅制參數轉為 tensor
    tax_params = torch.tensor(list(tax_params_dict.values()), dtype=torch.float32)
    tax_params = tax_params.repeat(required_batch_size, 1)

    # 建立 tensor
    moneydisposable_t = torch.tensor(moneydisposable, dtype=torch.float32)
    savings_t = torch.tensor(savings, dtype=torch.float32)
    v_t = torch.tensor(v, dtype=torch.float32)
    is_superstar_t = torch.tensor(is_superstar, dtype=torch.bool)
    tax_params_t = tax_params

    # 回傳字典，每項包含 value + shape
    return {
        "moneydisposable": {"value": moneydisposable_t, "shape": tuple(moneydisposable_t.shape)},
        "savings": {"value": savings_t, "shape": tuple(savings_t.shape)},
        "v": {"value": v_t, "shape": tuple(v_t.shape)},
        "is_superstar": {"value": is_superstar_t, "shape": tuple(is_superstar_t.shape)},
        "tax_params": {"value": tax_params_t, "shape": tuple(tax_params_t.shape)},
    }


In [109]:
state = initial_state(required_batch_size=256)

print(state["moneydisposable"]["shape"])
print(state["savings"]["shape"])
print(state["v"]["shape"])
print(state["is_superstar"]["shape"])
print(state["tax_params"]["shape"]) 


(256, 50)
(256, 50)
(256, 50)
(256, 50)
(256, 5)


In [110]:
def build_inputs(moneydisposable, savings, v, is_superstar, tax_params, carry_superstar=True):
    """
    回傳:
      features : (B, A, 2A + 2)          # 不含 is_superstar
      condi    : (B, A, Z)
      superstar: (B, A, 1) or None       # 只用來在輸出階段貼回，不進模型
    """
    B, A = moneydisposable.shape

    # (B, Z) -> (B, A, Z)
    condi = tax_params.unsqueeze(1).expand(-1, A, -1)

    # (B, 2A) -> (B, A, 2A)
    sum_info = torch.cat([moneydisposable, v], dim=1)         # (B, 2A)
    sum_info_rep = sum_info.unsqueeze(1).expand(-1, A, -1)    # (B, A, 2A)

    # (B, A, 1) × 2
    money_self   = moneydisposable.unsqueeze(-1)              # (B, A, 1)
    savings_self = savings.unsqueeze(-1)                      # (B, A, 1)

    features = torch.cat([sum_info_rep, money_self, savings_self], dim=2)  # (B, A, 2A+2)

    superstar = None
    if carry_superstar and is_superstar is not None:
        if is_superstar.dim() == 0:                # 標量 -> (B, A, 1)
            is_superstar = is_superstar.to(moneydisposable).view(1,1).expand(B, A).unsqueeze(-1)
        elif is_superstar.dim() == 2:              # (B, A) -> (B, A, 1)
            is_superstar = is_superstar.unsqueeze(-1)
        elif is_superstar.dim() == 3:              # (B, A, 1) 就保持
            pass
        else:
            raise ValueError("is_superstar must be scalar, (B,A), or (B,A,1)")

        # 若之後要與 y concat，確保 dtype 相容
        superstar = is_superstar.to(features.dtype)

    return features, condi, superstar


In [111]:
res2 = build_inputs(
    moneydisposable=state["moneydisposable"]["value"],
    savings=state["savings"]["value"],
    v = state["v"]["value"],
    is_superstar = state["is_superstar"]["value"],
    tax_params=state["tax_params"]["value"]
)

In [112]:
state_dim = 2*AGENTS + 2 # two state variables for each agent + 2 individual variables
cond_dim = 5 # exogenous variables for all agents in all worlds(Batch)
model = FiLMResNet2In(state_dim=state_dim, cond_dim=cond_dim,
                        hidden_dim=128, num_res_blocks=3, output_dim=2, dropout=0.1)

out = model(res2[0], res2[1])
at1, mut = torch.sigmoid(out[..., :1]), F.softplus(out[..., 1:]) + 1e-6
at1_squeezed, mut_squeezed = at1.squeeze(-1), mut.squeeze(-1)



# at1, mut = torch.sigmoid(at1), torch.exp(mut) # saving rate between 0 and 1 / ensure mu > 0
# print(res2[0].shape, res2[1].shape)
# print(model(res2[0], res2[1]).shape) # to log outputs
# print(flatten_last(model(res2[0], res2[1]))[0].shape) # to calculate loss